In [11]:
import os
import random
import string
import json
from PIL import Image, ImageDraw, ImageEnhance
from tqdm import tqdm
import numpy as np
import cv2


def draw_border_on_transparent_areasa_itoi(
    img,
    border_width=5,
    border_color=(0, 0, 255, 255)  # RGBA格式，默认红色
):
    # 提取Alpha通道（透明区域）
    if img.shape[2] == 4:
        alpha = img[:, :, 3]
    else:
        # 如果没有Alpha通道，整个图像视为不透明
        alpha = np.ones(img.shape[:2], dtype=np.uint8) * 255

    # 二值化Alpha通道（非透明区域=255，透明区域=0）
    _, mask = cv2.threshold(alpha, 1, 255, cv2.THRESH_BINARY)

    # 形态学膨胀（扩大非透明区域）
    kernel = np.ones((border_width, border_width), np.uint8)
    dilated = cv2.dilate(mask, kernel, iterations=1)

    # 计算边框区域（膨胀后的区域 - 原非透明区域）
    border_mask = dilated - mask

    # 创建纯色边框图像（RGBA）
    border_img = np.zeros_like(img)
    border_img[:, :, 0] = border_color[0]  # R
    border_img[:, :, 1] = border_color[1]  # G
    border_img[:, :, 2] = border_color[2]  # B
    border_img[:, :, 3] = border_mask      # A（仅在边框区域不透明）

    # 合并原图和边框
    result = img.copy()
    result[border_mask > 0] = border_img[border_mask > 0]

    return result


def transform_logo(logo, stretch_ratio=(1, 1), noise_intensity=0.05, color_factor=1.5):
    """
    对 logo 进行变换，包括添加噪点、按比例拉伸和改变颜色。

    :param logo: 原始 logo 图片
    :param stretch_ratio: 拉伸比例，元组形式 (width_ratio, height_ratio)
    :param noise_intensity: 噪点强度，取值范围 [0, 1]
    :param color_factor: 颜色变换因子，大于 1 增强颜色，小于 1 减弱颜色
    :return: 变换后的 logo 图片
    """
    # 按比例拉伸 logo
    width, height = logo.size
    new_width = int(width * stretch_ratio[0])
    new_height = int(height * stretch_ratio[1])
    logo = logo.resize((new_width, new_height), Image.Resampling.LANCZOS)

    # 添加噪点
    draw = ImageDraw.Draw(logo)
    for x in range(logo.width):
        for y in range(logo.height):
            if random.random() < noise_intensity:
                r, g, b, a = logo.getpixel((x, y))
                r = min(255, max(0, r + random.randint(-50, 50)))
                g = min(255, max(0, g + random.randint(-50, 50)))
                b = min(255, max(0, b + random.randint(-50, 50)))
                draw.point((x, y), (r, g, b, a))

    # 改变颜色
    enhancer = ImageEnhance.Color(logo)
    logo = enhancer.enhance(color_factor)

    return logo


def generate_logo_list(logo, K=30):
    '''随机变换logo，较耗时'''
    logo_list = [logo]
    for _ in tqdm(range(K)):
        stretch_ratio = (random.uniform(0.9, 1.1), random.uniform(0.9, 1.1))  # 拉伸比例
        noise_intensity = random.uniform(0.01, 0.1)  # 添加高斯噪音
        color_factor = random.uniform(0.8, 1.2)  # 改变颜色强度
        transformed_logo = transform_logo(
            logo.copy(), stretch_ratio, noise_intensity, color_factor)
        logo_list.append(transformed_logo)
    return logo_list


def randomize_hue(logo):
    """随机变换 logo 的色调，同时保留透明通道"""
    r, g, b, a = logo.split()
    hsv = Image.merge('RGB', (r, g, b)).convert('HSV')
    h, s, v = hsv.split()
    hue_shift = random.randint(-10, 10)
    h = h.point(lambda p: (p + hue_shift) % 256)
    rgb = Image.merge('HSV', (h, s, v)).convert('RGB')
    return Image.merge('RGBA', (*rgb.split(), a))


def overlay_logo_on_images(logo_name, target_folder, logo_path, output_folder, r=[0.05, 0.05], repeat=1, K=5, per_k=5):
    # 创建输出文件夹
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # 打开 logo 图片
    logo = Image.open(logo_path)

    # 缩小 logo 尺寸，使宽度和高度最大为 500，比例不变
    max_size = 200
    width, height = logo.size
    if width > max_size or height > max_size:
        ratio = min(max_size / width, max_size / height)
        new_width = int(width * ratio)
        new_height = int(height * ratio)
        logo = logo.resize((new_width, new_height), Image.Resampling.LANCZOS)

    # 生成 logo 列表
    logo_list = generate_logo_list(logo, K)
    # print('logo list', len(logo_list), logo_list)
    all_records = []

    # 遍历目标文件夹及其子目录
    print('target folder|', target_folder)
    for root, dirs, files in os.walk(target_folder):
        # print(root, dirs, files)
        for file in tqdm(files):
            if file.lower().endswith(('.jpg',)): # only jpg, png will error
                # 构建图片文件的完整路径
                image_path = os.path.join(root, file)
                # 打开图片
                image = Image.open(image_path)
                # 获取图片的宽度和高度
                image_width, image_height = image.size

                for _ in range(repeat):
                    # 生成定长随机文件名
                    random_name = ''.join(random.choices(
                        string.ascii_letters + string.digits, k=20)) + '.jpg'
                    output_path = os.path.join(output_folder, random_name)

                    image_copy = image.copy()
                    

                    # 随机决定叠加的 logo 数量
                    # num_logos = random.randint(1, per_k)
                    num_logos = per_k # 固定

                    image_records = {
                        "logo_name":logo_name,
                        "image": random_name,
                        "num": num_logos, 
                        "bounding boxes": [],
                        "segmentation_lists": []
                    }

                    for _ in range(num_logos):
                    # for _ in range(10):
                        # 随机选择 logo 宽度占图片宽度的比例
                        ratio = random.uniform(r[0], r[1])

                        # 从 logo 列表中随机选择一个 logo
                        selected_logo = random.choice(logo_list)

                        # 随机变换色调，同时保留透明通道
                        selected_logo = randomize_hue(selected_logo)

                        # 计算选择的 logo 的新宽度
                        logo_width = int(image_width * ratio)
                        # 按比例调整选择的 logo 的大小
                        logo_ratio = logo_width / selected_logo.width
                        new_height = int(selected_logo.height * logo_ratio)
                        logo_resized = selected_logo.resize(
                            (logo_width, new_height), Image.Resampling.LANCZOS)

                        # 随机确定 logo 的位置，确保 logo 完全在图片内
                        max_x = image_width - logo_width
                        max_y = image_height - new_height
                        x1 = random.randint(0, max_x)
                        y1 = random.randint(0, max_y)
                        x2 = x1 + logo_width
                        y2 = y1 + new_height

                        segmentation_logo = get_alpha_pixel_coords(logo_resized)
                        segmentation_tuple_list = [(pixel[0] + x1, pixel[1] + y1) for pixel in segmentation_logo]
                        segmentation_list = [coord for t in segmentation_tuple_list for coord in t]

                        # 将 logo 叠加到图片上
                        image_copy.paste(logo_resized, (x1, y1), logo_resized)

                        # 记录 bounding box 和 segmentation
                        image_records["bounding boxes"].append([x1, y1, x2, y2])
                        image_records["segmentation_lists"].append(segmentation_list)

                    # 保存处理后的图片
                    image_copy.save(output_path)

                    all_records.append(image_records)

    # 将记录写入 JSON 文件
    json_path = f'./generate_new/records/records_{logo_name}.json'
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    with open(json_path, 'w') as f:
        json.dump(all_records, f, indent=4)

    return all_records


###返回Alpha通道不为0的像素点的坐标
def get_alpha_pixel_coords(logo):
    image_array = np.array(logo)
    alpha_coords = []
    for i in range(image_array.shape[0]):
        for j in range(image_array.shape[1]):
            if image_array[i][j][3] != 0:
                alpha_coords.append((i, j))
    return alpha_coords    

In [12]:
import os
from tqdm import tqdm

dir_path = r"C:/目标检测数据集/evaluate/1_3_5" # 修改这里就能跑通
origin_folder = '/origin/'
logo_folder = '/logo/'
output_folder = '/generate_new/'


repeat = 1 # 每张背景图重复的次数
r = [0.10, 0.12] # logo宽度占背景图宽度的比例范围，均匀分布
K = 10 # 预先对logo进行变形，生成logo_list
# per_k = 5 # 背景图上叠加logo的数量，[1, per_k] 均匀分布


logo_list = os.listdir(dir_path + logo_folder)
print('logo list |', logo_list, '\n')
for logo_pic in logo_list:
    logo_name = logo_pic.split('.')[0]
    per_k = int(logo_name.split('_')[-1])
    print(logo_name, '|', logo_name)
    # break 
    origin_file_dir = dir_path + origin_folder + logo_name[:-2]
    output_file_dir = dir_path + output_folder + logo_name
    logo_file = dir_path + logo_folder + logo_pic
    records = overlay_logo_on_images(logo_name=logo_name, 
                                     target_folder=origin_file_dir, 
                                     logo_path=logo_file,
                                     output_folder=output_file_dir, 
                                     r=r,
                                     repeat=repeat, 
                                     K=K, 
                                     per_k=per_k)
    print("Records have been written to records.json.\n")


logo list | ['anker_1.png', 'anker_3.png', 'anker_5.png', 'fudan_pic_1.png', 'fudan_pic_3.png', 'fudan_pic_5.png', 'fudan_text_1.png', 'fudan_text_3.png', 'fudan_text_5.png', 'huawei_1.png', 'huawei_3.png', 'huawei_5.png', 'mcdonald_1.png', 'mcdonald_3.png', 'mcdonald_5.png'] 

anker_1 | anker_1


100%|██████████| 10/10 [00:00<00:00, 261.37it/s]


target folder| C:/目标检测数据集/evaluate/1_3_5/origin/anker


100%|██████████| 5/5 [00:00<00:00, 16.31it/s]


Records have been written to records.json.

anker_3 | anker_3


...

Records have been written to records.json.

mcdonald_5 | mcdonald_5


100%|██████████| 10/10 [00:00<00:00, 101.87it/s]


target folder| C:/目标检测数据集/evaluate/1_3_5/origin/mcdonald


100%|██████████| 5/5 [00:00<00:00,  5.28it/s]


Records have been written to records.json.



In [13]:
# import os
# from PIL import Image

# def convert_png_to_jpg(directory: str, quality: int = 95) -> None:
#     """
#     将指定目录下的所有PNG图片转换为JPG格式并删除源文件
    
#     参数:
#         directory (str): 目标目录路径
#         quality (int): JPG图片质量，范围1-100，默认95
#     """
#     if not os.path.exists(directory):
#         raise FileNotFoundError(f"目录不存在: {directory}")
    
#     for filename in os.listdir(directory):
#         if filename.lower().endswith('.png'):
#             # 构建完整文件路径
#             png_path = os.path.join(directory, filename)
            
#             # 跳过子目录
#             if not os.path.isfile(png_path):
#                 continue
                
#             # 构建输出JPG文件路径（替换扩展名）
#             jpg_filename = os.path.splitext(filename)[0] + '.jpg'
#             jpg_path = os.path.join(directory, jpg_filename)
            
#             try:
#                 # 打开并转换图像
#                 with Image.open(png_path) as img:
#                     # 处理RGBA模式（透明通道）
#                     if img.mode == 'RGBA':
#                         # 创建白色背景并合并
#                         rgb_img = Image.new('RGB', img.size, (255, 255, 255))
#                         rgb_img.paste(img, mask=img.split()[3])  # 3是Alpha通道
#                     else:
#                         rgb_img = img.convert('RGB')
                    
#                     # 保存为JPG
#                     rgb_img.save(jpg_path, 'JPEG', quality=quality)
#                     print(f"已转换: {png_path} -> {jpg_path}")
                
#                 # 删除原始PNG文件
#                 os.remove(png_path)
#                 print(f"已删除: {png_path}")
                
#             except Exception as e:
#                 print(f"处理文件 {png_path} 时出错: {str(e)}")
#                 continue

# # 使用示例
# convert_png_to_jpg("C:/目标检测数据集/evaluate/origin/fudan_text")

In [14]:
# import json
# import os

# # filte and save new data 

# logo_list = ['anker','fudan_pic','fudan_text'] 
# for logo_name in logo_list:
#     new_data = []
#     with open(r"C:\目标检测数据集\evaluate\generate_new\records\records_{}.json".format(logo_name), "r") as f:
#         data = json.load(f)
#     image_file_list = set(os.listdir(r"C:\目标检测数据集\evaluate\generate_new\{}".format(logo_name)))

#     for info in data:
#         if info['logo_name'] == logo_name and info['image'] in image_file_list:
#             new_data.append(info)
#     with open(r"C:\目标检测数据集\evaluate\generate_new\records\records_{}_filted.json".format(logo_name), "w") as f:
#         json.dump(new_data, f)



In [15]:
# import os 
# import json 


# logo_list = ['anker','fudan_pic','fudan_text'] 

# for logo_name in logo_list:
#     print(f'{"-"*10} {logo_name} {"-"*10}')
#     image_file_list = os.listdir(r"C:\目标检测数据集\evaluate\generate_new\{}".format(logo_name))
#     with open(r"C:\目标检测数据集\evaluate\generate_new\records\records_{}_filted.json".format(logo_name), "r") as f:
#         data = json.load(f)
#     image_data = [d['image'] for d in data]
#     print(f'json data:{len(image_data)}')
#     print(f'image file: {len(image_file_list)}')
#     for image_file in image_file_list:
#         if image_file not in image_data:
#             print(image_file)
